# MODEL VALIDATION – MAE & MAPE

This notebook independently validates Ridge Regression and Random Forest models 
using Mean Absolute Error (MAE) and Mean Absolute Percentage Error (MAPE).

The full preprocessing and time-aware split are recreated to ensure consistency.
No external model files are used.

In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
from sklearn.preprocessing import StandardScaler

print("Imports successful")

Imports successful


In [2]:
df = pd.read_csv("Final_fisheries_dataset.csv")

# Rename columns to match training structure
df = df.rename(columns={
    'country': 'Stock',
    'year': 'Year',
    'biomass_relative_to': 'Production'
})

df['Production'] = pd.to_numeric(df['Production'], errors='coerce')

print("Dataset loaded successfully")
print(df.head())

Dataset loaded successfully
                                           Stock  Year  Production    SST  \
0  Atlantic halibut Gulf of Maine / Georges Bank  1800    19795920  27.11   
1  Atlantic halibut Gulf of Maine / Georges Bank  1801    19795920  25.60   
2  Atlantic halibut Gulf of Maine / Georges Bank  1802    19795920  25.20   
3  Atlantic halibut Gulf of Maine / Georges Bank  1803    19795920  26.06   
4  Atlantic halibut Gulf of Maine / Georges Bank  1804    19775510  25.14   

   Trawling_Hours  country_encoded  
0         5638.58              100  
1         4840.09              100  
2         4709.12              100  
3         5335.08              100  
4         4890.73              100  


In [3]:
# Sort properly
df = df.sort_values(['Stock', 'Year'])

# Create Lag Features
df['Prod_Lag1'] = df.groupby('Stock')['Production'].shift(1)
df['SST_Lag1'] = df.groupby('Stock')['SST'].shift(1)

# Rolling Mean (Stability Feature)
df['Prod_3Y_Mean'] = df.groupby('Stock')['Production'].transform(lambda x: x.rolling(3).mean())

# SST Anomaly
df['SST_Mean'] = df.groupby('Stock')['SST'].transform('mean')
df['SST_Anomaly'] = df['SST'] - df['SST_Mean']

# Interaction Feature
df['Climate_Fishing_Pressure'] = df['SST'] * df['Trawling_Hours']

# Drop NA values created by lag/rolling
df = df.dropna()

print("Feature engineering completed")
print(df.head())

Feature engineering completed
                                            Stock  Year  Production    SST  \
701  Acadian redfish Gulf of Maine / Georges Bank  1914    23478190  26.01   
745  Acadian redfish Gulf of Maine / Georges Bank  1915    23478190  26.82   
795  Acadian redfish Gulf of Maine / Georges Bank  1916    23478190  26.26   
856  Acadian redfish Gulf of Maine / Georges Bank  1917    23478190  26.50   
919  Acadian redfish Gulf of Maine / Georges Bank  1918    23478190  26.64   

     Trawling_Hours  country_encoded   Prod_Lag1  SST_Lag1  Prod_3Y_Mean  \
701        17240.98                0  23478190.0     27.78  1.565213e+07   
745        17837.70                0  23478190.0     26.01  2.347819e+07   
795        17606.68                0  23478190.0     26.82  2.347819e+07   
856        17860.28                0  23478190.0     26.26  2.347819e+07   
919        18052.30                0  23478190.0     26.50  2.347819e+07   

      SST_Mean  SST_Anomaly  Climate_Fishing

In [4]:
# Define features used for modeling
features = [
    'Year',
    'SST',
    'Trawling_Hours',
    'Prod_Lag1',
    'SST_Lag1',
    'SST_Anomaly',
    'Climate_Fishing_Pressure'
]

X = df[features]
y = df['Production']

print("Features and target defined successfully")
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

Features and target defined successfully
Shape of X: (51578, 7)
Shape of y: (51578,)


In [5]:
# Time-Aware Split (same as training)

train_mask = df['Year'] < 2013

X_train = X[train_mask]
X_test = X[~train_mask]

y_train = y[train_mask]
y_test = y[~train_mask]

print("Time-aware split completed")
print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Time-aware split completed
Training samples: 49572
Testing samples: 2006


In [6]:
# ===============================
# MODEL TRAINING
# ===============================

scaler = StandardScaler()

# Scale for Ridge
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Ridge
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)

# Train Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

print("Models trained successfully")

Models trained successfully


In [7]:
# ===============================
# GENERATE PREDICTIONS
# ===============================

y_pred_ridge = ridge.predict(X_test_scaled)
y_pred_rf = rf.predict(X_test)

print("Predictions generated successfully")

Predictions generated successfully


In [8]:
# ===============================
# MAE & MAPE VALIDATION
# ===============================

ridge_mae = mean_absolute_error(y_test, y_pred_ridge)
rf_mae = mean_absolute_error(y_test, y_pred_rf)

ridge_mape = mean_absolute_percentage_error(y_test, y_pred_ridge)
rf_mape = mean_absolute_percentage_error(y_test, y_pred_rf)

print("MAE RESULTS")
print("------------")
print(f"Ridge MAE: {ridge_mae:.4f}")
print(f"Random Forest MAE: {rf_mae:.4f}")

print("\nMAPE RESULTS")
print("-------------")
print(f"Ridge MAPE: {ridge_mape:.4f}")
print(f"Random Forest MAPE: {rf_mape:.4f}")

MAE RESULTS
------------
Ridge MAE: 2243740.5461
Random Forest MAE: 2669195.8379

MAPE RESULTS
-------------
Ridge MAPE: 1292591170527180357632.0000
Random Forest MAPE: 854677490978024325120.0000


In [9]:
# ===============================
# FINAL VALIDATION METRICS
# ===============================

# Ridge Metrics
ridge_mae = mean_absolute_error(y_test, y_pred_ridge)
ridge_mape = mean_absolute_percentage_error(y_test, y_pred_ridge)

# Random Forest Metrics
rf_mae = mean_absolute_error(y_test, y_pred_rf)
rf_mape = mean_absolute_percentage_error(y_test, y_pred_rf)

print("\n" + "="*50)
print("        VALIDATION RESULTS")
print("="*50)

print("\nRidge Regression")
print(f"MAE  : {ridge_mae:.4f}")
print(f"MAPE : {ridge_mape:.4f}")

print("\nRandom Forest")
print(f"MAE  : {rf_mae:.4f}")
print(f"MAPE : {rf_mape:.4f}")


        VALIDATION RESULTS

Ridge Regression
MAE  : 2243740.5461
MAPE : 1292591170527180357632.0000

Random Forest
MAE  : 2669195.8379
MAPE : 854677490978024325120.0000


In [10]:
# ===============================
# VALIDATION SUMMARY TABLE
# ===============================

validation_results = pd.DataFrame({
    "Model": ["Ridge Regression", "Random Forest"],
    "MAE": [ridge_mae, rf_mae],
    "MAPE": [ridge_mape, rf_mape]
})

print("\nValidation Summary:")
display(validation_results.round(4))


Validation Summary:


,Model,MAE,MAPE
0,Ridge Regression,2.243741e+06,1.292591e+21
1,Random Forest,2.669196e+06,8.546775e+20


# FINAL CONCLUSION

- Ridge Regression achieves lower MAE compared to Random Forest.
- This indicates Ridge produces smaller average prediction errors.
- MAPE values are extremely large due to zero production values in the dataset.
- Since MAPE divides by actual values, zero targets cause instability.

Final Decision:
Ridge Regression performs better based on absolute error comparison (MAE).
For this dataset, MAE is a more reliable metric than MAPE.